## Calibrate value matrices to target cross-context Spearman correlations

**Goal**: for a fixed `base_width`, find the `delta_context` that produces a desired
average cross-context Spearman correlation.  Run a calibration sweep, interpolate the
`delta_context → correlation` curve, then generate a large batch of matrices at each
target correlation level and save them to `task_data/`.

Author: patrick.mccarthy@dpag.ox.ac.uk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from datetime import datetime
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.interpolate import interp1d

from cxval.tasks import ValueMatrix
from cxval.vis import STYLE

plt.style.use(STYLE)

## 0. Parameters

In [ ]:
# ── task structure ─────────────────────────────────────────────────────────
n_stimuli  = 5
n_contexts = 5
base_width = 0.5        # base values drawn from U(0.5 - w/2, 0.5 + w/2)

# ── calibration ────────────────────────────────────────────────────────────
n_calib_runs   = 100    # number of (base_seed, noise_seed) pairs for calibration
n_calib_points = 40     # resolution of delta_context sweep
delta_max      = 0.5    # upper bound of sweep (increase if curve does not reach target_corrs.min)

# ── targets ────────────────────────────────────────────────────────────────
target_corrs = np.round(np.arange(0.0, 1.01, 0.1), 2)  # [0.0, 0.1, ..., 1.0]

# ── large simulation ───────────────────────────────────────────────────────
n_large       = 200     # matrices per target correlation level
save_matrices = True    # if False, only params/seeds are saved (saves memory for very large grids)

# ── output ─────────────────────────────────────────────────────────────────
save_dir = Path("../task_data")
save_dir.mkdir(parents=True, exist_ok=True)

# ── derived ────────────────────────────────────────────────────────────────
stimuli    = [f"s{i}" for i in range(n_stimuli)]
contexts   = [f"c{j}" for j in range(n_contexts)]
base_lower = max(0.0, 0.5 - base_width / 2)
base_upper = min(1.0, 0.5 + base_width / 2)

print(f"n_stimuli={n_stimuli}  n_contexts={n_contexts}  base_width={base_width}")
print(f"base values drawn from U({base_lower:.2f}, {base_upper:.2f})")
print(f"target correlations: {target_corrs}")
print(f"n_large={n_large}  |  n_calib_runs={n_calib_runs}")

## 1. Calibration sweep

Sweep `delta_context` from 0 → `delta_max`, generate `n_calib_runs` value matrices at each
point, and record the average off-diagonal cross-context Spearman correlation.

In [ ]:
def avg_offdiag_spearman(mtx):
    """Mean Spearman r across all unique context-column pairs."""
    n_ctx = mtx.shape[1]
    rs = []
    for i in range(n_ctx):
        for j in range(i + 1, n_ctx):
            r, _ = spearmanr(mtx[:, i], mtx[:, j])
            rs.append(r)
    return float(np.mean(rs)) if rs else 1.0


delta_range = np.linspace(0.0, delta_max, n_calib_points)

rng_calib     = np.random.default_rng(0)
base_seeds_c  = rng_calib.integers(0, 100_000, size=n_calib_runs)
noise_seeds_c = rng_calib.integers(0, 100_000, size=n_calib_runs)

calib_corrs = np.zeros((n_calib_runs, n_calib_points))

for ri, (bseed, nseed) in enumerate(zip(base_seeds_c, noise_seeds_c)):
    vm = ValueMatrix(
        seed=int(bseed), contexts=contexts, stimuli=stimuli,
        delta_context=0.0, base_lower=base_lower, base_upper=base_upper,
    )
    vm.generate_base_values(seed=int(bseed))
    for di, dc in enumerate(delta_range):
        vm.delta_context = float(dc)
        mtx = vm.generate_value_matrix(seed=int(nseed) + di)
        calib_corrs[ri, di] = avg_offdiag_spearman(mtx)

calib_mean = calib_corrs.mean(axis=0)
calib_sem  = calib_corrs.std(axis=0) / np.sqrt(n_calib_runs)

print(f"Calibration done.")
print(f"Correlation range covered: [{calib_mean.min():.3f}, {calib_mean.max():.3f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for ri in range(n_calib_runs):
    ax.plot(delta_range, calib_corrs[ri], color="gray", alpha=0.15, linewidth=0.6)
ax.fill_between(delta_range, calib_mean - calib_sem, calib_mean + calib_sem,
                color="steelblue", alpha=0.3)
ax.plot(delta_range, calib_mean, color="steelblue", linewidth=2, label="mean ± SEM")
ax.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_xlabel("delta_context")
ax.set_ylabel("Avg cross-context Spearman r")
ax.set_title(
    f"Calibration curve  "
    f"(base_width={base_width}, {n_stimuli}s × {n_contexts}c, {n_calib_runs} runs)"
)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Interpolation

Invert the calibration curve to obtain `target_corr → delta_context`.
Because the mean curve may have small non-monotonicities (variance from small n_stimuli),
we sort by correlation value before interpolating.

In [ ]:
# Sort calibration points so correlation is strictly increasing (for interp1d)
# — curve goes from high (delta=0, r≈1) to low (delta=delta_max).
sort_idx   = np.argsort(calib_mean)          # ascending corr order
corr_asc   = calib_mean[sort_idx]
delta_asc  = delta_range[sort_idx]

# Deduplicate (rare but possible with very flat curves)
_, uniq = np.unique(corr_asc, return_index=True)
corr_asc  = corr_asc[uniq]
delta_asc = delta_asc[uniq]

# Interpolant: corr → delta_context
interp_fn = interp1d(
    corr_asc, delta_asc, kind="linear",
    bounds_error=False,
    fill_value=(delta_asc[0], delta_asc[-1]),
)

achievable_min = float(corr_asc[0])
achievable_max = float(corr_asc[-1])

print(f"Achievable correlation range: [{achievable_min:.3f}, {achievable_max:.3f}]")
print(f"Targets outside this range will be clipped to the boundary delta_context.\n")

target_deltas = {}
for tc in target_corrs:
    td = float(np.clip(interp_fn(tc), 0.0, delta_max))
    target_deltas[tc] = td
    flag = ""
    if tc < achievable_min - 0.01 or tc > achievable_max + 0.01:
        flag = "  ← outside calibration range"
    print(f"  target r={tc:.1f}  →  delta_context={td:.4f}{flag}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(delta_range, calib_mean - calib_sem, calib_mean + calib_sem,
                color="steelblue", alpha=0.3)
ax.plot(delta_range, calib_mean, color="steelblue", linewidth=2, label="calibration (mean ± SEM)")

target_colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(target_corrs)))
for tc, color in zip(target_corrs, target_colors):
    td = target_deltas[tc]
    ax.scatter(td, tc, color=color, zorder=5, s=60)
    ax.annotate(f"{tc:.1f}", (td, tc), textcoords="offset points",
                xytext=(5, 0), fontsize=7, color=color)

ax.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_xlabel("delta_context")
ax.set_ylabel("Avg cross-context Spearman r")
ax.set_title("Interpolated target correlations on calibration curve")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Large simulation

For each target correlation, generate `n_large` value matrices using the
interpolated `delta_context`.  Each matrix gets a fresh `base_seed`.

In [ ]:
rng_large    = np.random.default_rng(42)
base_seeds_l = rng_large.integers(0, 1_000_000, size=n_large)
noise_seed_l = 99   # fixed noise seed so the only variation is in base values

matrices        = {}   # target_corr → list of (n_stimuli, n_contexts) arrays
actual_corrs    = {}   # target_corr → array of per-matrix avg Spearman

for tc in target_corrs:
    dc = target_deltas[tc]
    mats, acs = [], []
    for bseed in base_seeds_l:
        vm = ValueMatrix(
            seed=int(bseed), contexts=contexts, stimuli=stimuli,
            delta_context=dc, base_lower=base_lower, base_upper=base_upper,
        )
        vm.generate_base_values(seed=int(bseed))
        mtx = vm.generate_value_matrix(seed=noise_seed_l)
        mats.append(mtx)
        acs.append(avg_offdiag_spearman(mtx))
    matrices[tc]     = mats
    actual_corrs[tc] = np.array(acs)
    print(f"  r={tc:.1f}  delta={dc:.4f}  actual mean={actual_corrs[tc].mean():.3f} ± {actual_corrs[tc].std():.3f}")

print("\nDone.")

## 4. Verification

In [ ]:
actual_means = np.array([actual_corrs[tc].mean() for tc in target_corrs])
actual_stds  = np.array([actual_corrs[tc].std()  for tc in target_corrs])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# left: target vs actual
ax = axes[0]
ax.errorbar(target_corrs, actual_means, yerr=actual_stds,
            fmt="o", color="steelblue", capsize=4, label="actual (mean ± SD)")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.6, label="ideal")
ax.set_xlabel("Target cross-context Spearman r")
ax.set_ylabel("Actual mean Spearman r")
ax.set_title("Target vs actual correlation")
ax.legend(fontsize=8)

# right: distributions per target
ax2 = axes[1]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(target_corrs)))
for tc, color in zip(target_corrs, colors):
    vals = actual_corrs[tc]
    ax2.scatter(
        np.full(len(vals), tc) + np.random.default_rng(0).uniform(-0.03, 0.03, len(vals)),
        vals, color=color, alpha=0.3, s=8,
    )
    ax2.scatter(tc, vals.mean(), color=color, s=60, zorder=5, edgecolors="k", linewidths=0.5)

ax2.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.6)
ax2.set_xlabel("Target cross-context Spearman r")
ax2.set_ylabel("Per-matrix Spearman r")
ax2.set_title(f"Distribution across {n_large} matrices per target")

plt.tight_layout()
plt.show()

In [ ]:
# example matrices for each target correlation level
n_targets = len(target_corrs)
fig, axes = plt.subplots(1, n_targets, figsize=(2.2 * n_targets, 2.5))

for ax, tc in zip(axes, target_corrs):
    ex_mtx = matrices[tc][0]   # first matrix at this level
    im = ax.imshow(ex_mtx, cmap="hot", vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"r={tc:.1f}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

axes[0].set_ylabel("Stimulus")
fig.colorbar(im, ax=axes[-1], shrink=0.8, label="Value")
plt.suptitle("Example value matrix per target correlation level", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Save

In [ ]:
timestamp = datetime.now().strftime("%d_%m_%y")
fname = (
    f"{timestamp}_calibrated_vm"
    f"_{n_stimuli}s{n_contexts}c"
    f"_bw{base_width:.2f}"
    f"_n{n_large}.pkl"
)
save_path = save_dir / fname

payload = {
    "params": {
        "n_stimuli":         n_stimuli,
        "n_contexts":        n_contexts,
        "base_width":        base_width,
        "base_lower":        base_lower,
        "base_upper":        base_upper,
        "target_corrs":      target_corrs,
        "target_deltas":     target_deltas,
        "n_large":           n_large,
        "noise_seed":        noise_seed_l,
        "base_seeds":        base_seeds_l,
    },
    "calibration": {
        "delta_range":   delta_range,
        "calib_mean":    calib_mean,
        "calib_sem":     calib_sem,
        "calib_corrs":   calib_corrs,   # (n_calib_runs, n_calib_points)
    },
    "actual_corrs":  actual_corrs,      # target_corr → (n_large,) array
}

if save_matrices:
    payload["matrices"] = {
        tc: np.stack(mats).astype(np.float32)   # (n_large, n_stimuli, n_contexts)
        for tc, mats in matrices.items()
    }

with open(save_path, "wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved → {save_path}")
if save_matrices:
    total_mats = n_large * len(target_corrs)
    print(f"Stored {total_mats} matrices  ({n_large} × {len(target_corrs)} targets)")
else:
    print("Matrices not saved (save_matrices=False); params and seeds stored only.")

### Loading example

```python
import pickle
with open(save_path, "rb") as f:
    data = pickle.load(f)

# (n_large, n_stimuli, n_contexts) array of matrices at target r=0.5
mats_r05 = data["matrices"][0.5]

# delta_context used for target r=0.3
dc_r03 = data["params"]["target_deltas"][0.3]
```